# Projects, data retention and metrics

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Governance on `bedrock-mantle`: separating workloads with Projects, attributing cost to
them, controlling what AWS retains, and finding the metrics. Authentication and URL
paths are in
[`01-endpoints-auth-and-paths`](01-endpoints-auth-and-the-three-paths.ipynb).

**Covered:** Projects, attributing inference to a project from each API, project tags,
`store` and conversation state, data retention modes per model, and CloudWatch metrics.

## Setup

Creating projects needs more than inference access: `AmazonBedrockMantleFullAccess`
covers it, and `AmazonBedrockMantleInferenceAccess` alone can read projects but not
create them. The management operations are plain HTTPS calls, which `post` from
`_shared/bedrock.py` signs; it returns `(status, body)` instead of raising. `redact_ids`
shortens opaque IDs before they reach this committed output.

In [1]:
import json
import sys

import anthropic
from aws_bedrock_token_generator import provide_token
from openai import BadRequestError, NotFoundError, OpenAI

sys.path.insert(0, "../_shared")
from bedrock import err, post, redact_ids

REGION = "us-east-1"
MANTLE = f"https://bedrock-mantle.{REGION}.api.aws"

code_, listing = post("/v1/organization/projects", None, region=REGION, method="GET")
projects = listing.get("data", [])
# The listing returns active projects only, so an "active" tally would restate
# len(projects). Section 7 archives a project and checks that it drops out.
print(f"GET /v1/organization/projects -> {code_}: {len(projects)} listed, "
      f"default present: {any(p.get('id') == 'default' for p in projects)}, "
      f"{sum(1 for p in projects if p.get('tags'))} tagged")

GET /v1/organization/projects -> 200: 9 listed, default present: True, 8 tagged


## 1. Projects

A Project is the boundary for one workload, such as an application or an environment. It
gives IAM-enforceable isolation, cost tracking through tags, and per-project metrics
without separate AWS accounts. Every account has a `default` project, and an account can
have up to 1,000. With the Anthropic Messages API the same resource is called a
Workspace. Set tags when you create a project, because they are what Cost Explorer
groups by. The `arn` in the reply is what an IAM policy names to scope access to the
project.

In [2]:
# An interrupted earlier run can leave this demo project behind. Everything the
# listing returns is active, so matching on the name is the whole test.
for leftover in projects:
    if leftover.get("name", "").startswith("samples-foundations"):
        post(f"/v1/organization/projects/{leftover['id']}/archive", {}, region=REGION)
        print("archived a leftover demo project:", leftover["name"])

code_, project = post("/v1/organization/projects", {
    "name": "samples-foundations",
    "tags": {"Application": "Samples", "Environment": "Demo", "CostCenter": "0000"},
}, region=REGION)
if code_ != 200:
    raise RuntimeError(f"create failed: HTTP {code_} {err(project)}")

PROJECT_ID = project["id"]
print("created :", redact_ids(PROJECT_ID), "| status:", project["status"])
print("tags    :", project["tags"])
print("retention mode:", project["data_retention"]["mode"])

created : proj_szateeax... | status: active
tags    : {'CostCenter': '0000', 'Application': 'Samples', 'Environment': 'Demo'}
retention mode: inherit


## 2. Attributing inference to a project

Each API names the project in a different header: `OpenAI-Project` for Responses and
Chat Completions, and `anthropic-workspace` for Messages. The OpenAI SDK sets its header
from the `project` argument. Usage attributed this way rolls up under the project's
tags.

In [3]:
key = provide_token(region=REGION)
responses_client = OpenAI(base_url=MANTLE + "/openai/v1", api_key=key, project=PROJECT_ID, timeout=120)
chat_client = OpenAI(base_url=MANTLE + "/v1", api_key=key, project=PROJECT_ID, timeout=120)
messages_client = anthropic.Anthropic(base_url=MANTLE + "/anthropic", auth_token=key, api_key=None,
                                      default_headers={"anthropic-workspace": PROJECT_ID}, timeout=120)
QUESTION = "What is the capital of Australia? Answer in one word."

r = responses_client.responses.create(model="google.gemma-4-31b", input=QUESTION, max_output_tokens=50, store=False)
c = chat_client.chat.completions.create(model="qwen.qwen3-32b", messages=[{"role": "user", "content": QUESTION}],
                                        max_tokens=500)
m = messages_client.messages.create(model="anthropic.claude-haiku-4-5", max_tokens=50,
                                    messages=[{"role": "user", "content": QUESTION}])
print("Responses        (OpenAI-Project)     :", r.output_text.strip())
print("Chat Completions (OpenAI-Project)     :", (c.choices[0].message.content or "").strip())
print("Messages         (anthropic-workspace):", "".join(b.text for b in m.content if b.type == "text").strip())

Responses        (OpenAI-Project)     : Canberra
Chat Completions (OpenAI-Project)     : Canberra
Messages         (anthropic-workspace): Canberra


## 3. Tags and names

Tags are added and removed by key; no operation replaces the whole set at once. The same
call renames a project.

In [4]:
path = f"/v1/organization/projects/{PROJECT_ID}"
code_, updated = post(path, {"add_tags": {"Team": "Samples", "Version": "1.0"}}, region=REGION)
print("add_tags        ->", code_, sorted(updated.get("tags", {})))
code_, updated = post(path, {"remove_tag_keys": ["Version"]}, region=REGION)
print("remove_tag_keys ->", code_, sorted(updated.get("tags", {})))
code_, updated = post(path, {"name": "samples-foundations-v2"}, region=REGION)
print("rename          ->", code_, updated.get("name"))

add_tags        -> 200 ['Application', 'CostCenter', 'Environment', 'Team', 'Version']
remove_tag_keys -> 200 ['Application', 'CostCenter', 'Environment', 'Team']


rename          -> 200 samples-foundations-v2


## 4. `store` and conversation state

On the Responses API, `store` defaults to `true`, which retains the input and output for
30 days, encrypted, in the Region and project that served them. That retention is what
lets a later call continue with `previous_response_id`. With `store=False` nothing is
kept, and continuing from that response fails with 400 `previous_response_not_found`.
Gemma 4, used here, returned 404 `Response not found.` instead in some September 2026
runs, so handle both. If you need both privacy and several turns, send the history
yourself on every call.

In [5]:
kept = responses_client.responses.create(model="google.gemma-4-31b", input="My name is Ada. Reply OK.",
                                         max_output_tokens=50, store=True)
unkept = responses_client.responses.create(model="google.gemma-4-31b", input="My name is Ada. Reply OK.",
                                           max_output_tokens=50, store=False)

follow_up = responses_client.responses.create(model="google.gemma-4-31b", previous_response_id=kept.id,
                                              input="What is my name? One word.", max_output_tokens=50,
                                              store=False)
print("from the stored response  :", follow_up.output_text.strip())
try:
    responses_client.responses.create(model="google.gemma-4-31b", previous_response_id=unkept.id,
                                      input="What is my name? One word.", max_output_tokens=50, store=False)
except (BadRequestError, NotFoundError) as exc:  # 400 or 404, depending on the day
    body = exc.body if isinstance(exc.body, dict) else {}
    print("from the unstored response:", exc.status_code, body.get("code") or "", redact_ids(body.get("message") or ""))

responses_client.responses.delete(kept.id)
print("deleted the stored response")

from the stored response  : Ada
from the unstored response: 400 previous_response_not_found Previous response with id 'resp_vfl6w45a...' not found.


deleted the stored response


## 5. Data retention modes

Separately from `store`, a data retention mode decides whether AWS retains content, for
example content a classifier flags for review. A mode of `inherit` defers to the next
scope up: project, then account, then the model's own default, and `source` in the reply
says which scope decided.

As of October 2026 the [data retention
page](https://docs.aws.amazon.com/bedrock/latest/userguide/data-retention.html) of the
Bedrock User Guide lists `inherit`, covered above, and four modes that form an ordered
scale, least to most permissive below. What each one permits is from that page; the cell
below reads back which of them each model accepts.

| Mode | What it permits |
|---|---|
| `none` | Zero data retention: no input or output is written to durable storage, so a response cannot be kept for a later call to continue from. |
| `default` | The model's own policy applies. AWS may retain content for abuse detection, and the model provider does not receive it. |
| `aws_review` | AWS retains inputs and outputs for up to 30 days and may review them by hand, within the AWS boundary. Some providers require that review as a condition of access to their models. |
| `provider_data_share` | Legacy, and the most permissive of the four. Bedrock does not share content with model providers today, so it behaves as `aws_review`; set `aws_review` instead in a new configuration. |

A model is callable when the effective mode is at or above the mode that model requires,
and `allowed_modes` holds the modes that satisfy it, so its least permissive entry is
that model's minimum. `GET /v1/models/<id>` reports a model whose minimum sits above the
effective mode as `status: unavailable`, with a `status_reason` naming the mode in force,
so you can read the gate off the catalogue instead of discovering it on a call.

The output below has `aws_review` in all three sets and `none` missing from
`openai.gpt-5.6-sol`, which therefore has no zero-retention posture in this account. Read
that absence as an account-level fact rather than a property of the model: the page's
zero data retention section says that for a model which requires retention, `none`
appears in `allowed_modes` for accounts approved for zero retention on that model, and
that eligibility is evaluated per account and per model. So if you need zero retention on
a model that does not list it, ask your AWS account manager, or Anthropic for a Claude
model, rather than reading the absence as final. The same page notes that a permissive setting retains no more than each
model allows: a model whose `allowed_modes` includes `none` is not retained whatever the
scope above it says.

The mode is set per endpoint as well as per Region, and the two settings are independent.
`PUT /v1/data_retention` sets it for `bedrock-mantle`; for `bedrock-runtime` the same page
gives `PUT /data-retention` on `bedrock.<Region>.amazonaws.com`, and neither setting reads
the other. Read both with the matching GET before you conclude anything about a gated
model. In `us-east-1` in October 2026 the `bedrock-mantle` setting in this account read
back `{"mode": "inherit"}`, which fell through to the model default: Claude Fable 5
allows only `aws_review` and `provider_data_share`, so under the resulting `default` it
reported `status: unavailable` and the call returned 400 `data retention mode 'default'
is not available for this model`. The same prompt through `us.anthropic.claude-fable-5`
on `bedrock-runtime` answered, because that endpoint reads the control-plane setting,
which in this account was at or above `aws_review`. That is one rule applied to two
independent settings rather than an endpoint that skips the gate; the page says an
account whose control-plane mode is below the minimum is refused on `bedrock-runtime`
too, with a `ValidationException`. Set the mode on both endpoints if your posture has to
hold for both, and note that reading the control-plane one needs
`bedrock:GetAccountDataRetention`, which an identity allowed to call the model may not
have. This notebook only reads the modes, because changing an account's mode is an
account-wide change.

In [6]:
code_, account = post("/v1/data_retention", None, region=REGION, method="GET")
print("account mode:", account.get("mode"))

models = OpenAI(base_url=MANTLE + "/v1", api_key=key, timeout=120).models
for model_id in ("google.gemma-4-31b", "openai.gpt-5.6-sol", "anthropic.claude-haiku-4-5"):
    retention = getattr(models.retrieve(model_id), "data_retention", None) or {}
    allowed = retention.get("allowed_modes", [])
    print(f"{model_id:28} mode={retention.get('mode')!s:8} source={retention.get('source')!s:14} "
          f"allowed={sorted(allowed)}  zero retention possible: {'none' in allowed}")

account mode: inherit
google.gemma-4-31b           mode=default  source=model_default  allowed=['aws_review', 'default', 'none', 'provider_data_share']  zero retention possible: True


openai.gpt-5.6-sol           mode=default  source=model_default  allowed=['aws_review', 'default', 'provider_data_share']  zero retention possible: False


anthropic.claude-haiku-4-5   mode=default  source=model_default  allowed=['aws_review', 'default', 'none', 'provider_data_share']  zero retention possible: True


A mode stays in place only while nobody moves it: any identity holding the write action
can set another, and the read-back above says nothing about who may change it. As of
October 2026 the same page's enforcement section gives a condition key on each write
action, so a service control policy can bound which modes anyone in an organisation may
set: `bedrock-mantle:DataRetentionMode` on `bedrock-mantle:PutAccountDataRetention`,
`bedrock-mantle:CreateProject` and `bedrock-mantle:UpdateProject`, and
`bedrock:DataRetentionMode` on `bedrock:PutAccountDataRetention`. Each service's action
carries only its own key, so each needs its own statement, as the bearer-token policy in
section 8 of [`01-endpoints-auth-and-paths`](01-endpoints-auth-and-the-three-paths.ipynb)
does. To pin an account to zero retention, deny the write unless the mode is `none`:

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "PinMantleRetentionToNone",
      "Effect": "Deny",
      "Action": [
        "bedrock-mantle:PutAccountDataRetention",
        "bedrock-mantle:CreateProject",
        "bedrock-mantle:UpdateProject"
      ],
      "Resource": "*",
      "Condition": {"StringNotEquals": {"bedrock-mantle:DataRetentionMode": "none"}}
    },
    {
      "Sid": "PinControlPlaneRetentionToNone",
      "Effect": "Deny",
      "Action": "bedrock:PutAccountDataRetention",
      "Resource": "*",
      "Condition": {"StringNotEquals": {"bedrock:DataRetentionMode": "none"}}
    }
  ]
}
```

A negated condition is true when the key is absent, which the IAM reference on [condition
operators](https://docs.aws.amazon.com/IAM/latest/UserGuide/reference_policies_elements_condition_operators.html)
states for condition operators other than `...IfExists` and `Null`. Two things follow for
a `Deny` on `StringNotEquals`. A single statement naming both services' actions would
carry one key only, and the other service's write would then be denied whatever mode it
set, which is why the policy above has a statement per service. A write that carries no
mode at all is denied as well. That catches the project create in section 1, which sends
no `data_retention` block, and all three project updates in section 3, `add_tags`,
`remove_tag_keys` and the rename, none of which has a mode to set at all. Restrict the
`Deny` to writes that do carry one by adding a presence test, since each condition block
in a statement has to be true for the `Deny` to apply:

```json
"Condition": {
  "StringNotEquals": {"bedrock-mantle:DataRetentionMode": "none"},
  "Null": {"bedrock-mantle:DataRetentionMode": "false"}
}
```

`Null` with `false` reads as "the key is present", so a tag update or a rename passes
while an explicit `aws_review` is still denied. Only the project actions need it: `mode`
is a required field on both account writes, and a `PUT /v1/data_retention` that omits it
is refused with 400 `missing field \`mode\`` (October 2026), so the key is present in the
request context there and the plain `StringNotEquals` form holds. The cost of the
presence test is that a create carrying no mode now passes as well, which is the
inheritance gap below and the reason to pin the account scope too.

To allow retention for abuse detection but not human review, the same section inverts the
test: deny the write when the mode is `aws_review` or `provider_data_share`, under
`ForAnyValue:StringEquals`. That leaves `none`, `default` and `inherit` settable. A
positive test is false when the key is absent, where the negated form above is true, so
this shape does not fire on a write that carries no mode, and such a write leaves the
scope at `inherit`: the create in section 1 reads back `retention mode: inherit`. What
the policy bounds is therefore what an identity may explicitly set, not the effective
mode, which falls through `inherit` to the next scope up and ends at a model default that
is read-only and that no condition key reaches. Pin the account scope to a concrete mode
as well if the posture has to hold for a gated model, and read both scopes back with the
GETs above.

The key sits on the write path, so a policy built on it neither grants nor blocks
inference: the retention gate above is what refuses a call to a gated model. Attaching
one also leaves a mode already configured where it is rather than resetting it, so read
both scopes back with the GETs above once it is in place.

## 6. Metrics and logs

`bedrock-mantle` publishes CloudWatch metrics in the `AWS/BedrockMantle` namespace. A
dashboard built on `AWS/Bedrock`, where `bedrock-runtime` publishes, shows nothing for
mantle traffic. The only error metric in `AWS/BedrockMantle` is `InferenceClientErrors`,
which counts 4xx responses, so watch 5xx errors in your own client telemetry. CloudTrail
records mantle inference as data events, which are off by default and billed; creating
projects appears as management events. Minting a short-term API key happens on the
client and is not logged. In CloudFormation a project is `AWS::BedrockMantle::Project`.

In [7]:
import boto3

cloudwatch = boto3.client("cloudwatch", region_name=REGION)
for namespace in ("AWS/BedrockMantle", "AWS/Bedrock"):
    names = sorted({m["MetricName"] for page in cloudwatch.get_paginator("list_metrics").paginate(Namespace=namespace)
                    for m in page["Metrics"]})
    print(f"{namespace:18} {len(names):2} metrics: {', '.join(names[:8])}{' …' if len(names) > 8 else ''}")

AWS/BedrockMantle   8 metrics: BurnDownConsumed, EquivalentReservationUnits, InferenceClientErrors, Inferences, InputTokens, OutputTokens, TotalInputTokens, TotalOutputTokens
AWS/Bedrock        14 metrics: CacheReadInputTokenCount, CacheWriteInputTokenCount, EstimatedTPMQuotaUsage, InputTokenCount, InvocationClientErrors, InvocationLatency, InvocationServerErrors, InvocationThrottles …


## 7. Clean up

Archiving a project stops new inference against it, and its data stays readable for
about 30 days. `GET /v1/organization/projects` returns active projects only, so an
archived project leaves the listing straight away. Measured in US East (N. Virginia)
in October 2026, a `?status=archived` query returned the same active list rather than
an error, so the listing is not the way to reach one. Its own URL is:
`GET /v1/organization/projects/{id}` returned `status: archived` with an `archived_at`
timestamp and the tags intact. Keep the project ID if you will want that history,
because the listing will not hand it back.

In [8]:
code_, archived = post(f"/v1/organization/projects/{PROJECT_ID}/archive", {}, region=REGION)
print("archive ->", code_, archived.get("status"))

code_, after = post("/v1/organization/projects", None, region=REGION, method="GET")
print("still in the listing:", any(p["id"] == PROJECT_ID for p in after.get("data", [])))

# An archived project keeps its own URL, so the ID is the way back to it.
code_, one = post(f"/v1/organization/projects/{PROJECT_ID}", None, region=REGION, method="GET")
print("GET by ID ->", code_, one.get("status"), "| archived_at set:", "archived_at" in one)
print("tags after archiving:", sorted(one.get("tags", {})))

archive -> 200 archived
still in the listing: False
GET by ID -> 200 archived | archived_at set: True
tags after archiving: ['Application', 'CostCenter', 'Environment', 'Team']


## Things that differ

What trips people up here, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Project header | Usage lands in `default` | Send `OpenAI-Project` for Responses and Chat Completions, `anthropic-workspace` for Messages (section 2). |
| `store` defaults to `true` | Responses are retained for 30 days | Pass `store=False` when you must not retain, and send history yourself (section 4). |
| Continuing an unstored response | 400 `previous_response_not_found`, or 404 `Response not found.` | Store it, or resend the history. |
| A model that needs a more permissive retention mode | 400 `data retention mode 'default' is not available for this model` on `bedrock-mantle`, and `status: unavailable` for that model in the catalogue | Read the model's `allowed_modes`, then set a mode it allows at the scope the endpoint reads: the project or `PUT /v1/data_retention` for `bedrock-mantle`, `PUT /data-retention` on `bedrock.<Region>.amazonaws.com` for `bedrock-runtime` (section 5). |
| A retention mode that nothing pins | A later `GET` returns a mode nobody recorded setting | Deny the write unless the mode matches your posture, with `bedrock-mantle:DataRetentionMode` on the mantle write actions and `bedrock:DataRetentionMode` on `bedrock:PutAccountDataRetention`, one statement per service (section 5). |
| CloudWatch namespace | Empty `AWS/Bedrock` graphs for mantle traffic | Use `AWS/BedrockMantle`. |
| No 5xx metric | Only `InferenceClientErrors` (4xx) | Record server errors in your own client. |
| Finding an archived project | It drops out of `GET /v1/organization/projects`, and `?status=archived` returns the same active list | Keep the ID: `GET /v1/organization/projects/{id}` still returns it with `status: archived` (section 7). |
| Tag updates | No call replaces every tag at once | Use `add_tags` and `remove_tag_keys`. |

## Next

- Quotas, retries, service tiers and latency:
  [`03-scaling-tiers-and-latency`](03-scaling-tiers-and-latency.ipynb)
- Converse and inference profiles on `bedrock-runtime`:
  [`04-bedrock-runtime-converse-and-profiles`](04-bedrock-runtime-converse-and-profiles.ipynb)
- Production hardening across families:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)